In [ ]:
!pip install tensorflow==2.15

In [ ]:
import tensorflow as tf

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle

In [ ]:
!kaggle datasets download -d tawsifurrahman/covid19-radiography-database


Dataset URL: https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database
License(s): copyright-authors
 99% 771M/778M [00:12<00:00, 116MB/s] 
100% 778M/778M [00:12<00:00, 63.1MB/s]


In [ ]:
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!unzip '/content/covid19-radiography-database.zip'

Streaming output truncated to the last 5000 lines.
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-7921.png  
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-7922.png  
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-7923.png  
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-7924.png  
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-7925.png  
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-7926.png  
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-7927.png  
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-7928.png  
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-7929.png  
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-793.png  
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-7930.png  
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-7931.png  
  inflating: COVID-19_Radiography_Dataset/Normal/masks/Normal-7932.png  
 

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import shutil

# Define the path to the unzipped dataset
dataset_path = '/content/COVID-19_Radiography_Dataset/'

# Create directories for train and test splits
train_dir = '/content/train'
test_dir = '/content/test'
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Create subdirectories for each class in train and test directories
classes = ['COVID', 'NORMAL', 'VIRAL_PNEUMONIA', 'LUNG_OPACITY']
for cls in classes:
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)

# Function to get all image file paths from the directory
def get_image_paths(directory):
    image_paths = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                image_paths.append(os.path.join(root, file))
    return image_paths

# Function to split the data
def split_data(SOURCE, TRAINING, TESTING, SPLIT_SIZE):
    data = get_image_paths(SOURCE)
    if not data:
        print(f'No images found in {SOURCE}')
        return

    train_data, test_data = train_test_split(data, test_size=1-SPLIT_SIZE, random_state=42)

    for item in train_data:
        shutil.copy(item, os.path.join(TRAINING, os.path.basename(item)))

    for item in test_data:
        shutil.copy(item, os.path.join(TESTING, os.path.basename(item)))

# Split the dataset
split_size = 0.8
for cls in classes:
    SOURCE_DIR = os.path.join(dataset_path, cls)
    TRAIN_DIR = os.path.join(train_dir, cls)
    TEST_DIR = os.path.join(test_dir, cls)
    split_data(SOURCE_DIR, TRAIN_DIR, TEST_DIR, split_size)

# Now you have train and test directories with respective class subdirectories and images




No images found in /content/COVID-19_Radiography_Dataset/NORMAL
No images found in /content/COVID-19_Radiography_Dataset/VIRAL_PNEUMONIA
No images found in /content/COVID-19_Radiography_Dataset/LUNG_OPACITY


In [ ]:
trainPath = '/content/train'
testPath = '/content/test'

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255,zoom_range=0.2,shear_range=0.2)
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train = train_datagen.flow_from_directory(trainPath,target_size=(224,224),batch_size=16)
test = test_datagen.flow_from_directory(testPath,target_size=(224,224),batch_size=16)

Found 3476 images belonging to 4 classes.
Found 1307 images belonging to 4 classes.


# VGG16

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.layers import Dense,Flatten
from tensorflow.keras.models import Model

In [ ]:
vgg = VGG16(include_top=False,input_shape=(224,224,3))

58889256/58889256 [==============================] - 0s 0us/step


In [ ]:
vgg.summary()

Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [ ]:
for layer in vgg.layers:
  print(layer)

In [ ]:
for layer in vgg.layers:
  layer.trainable=False

In [ ]:
x = Flatten()(vgg.output)

In [ ]:
output = Dense(4,activation='softmax')(x)

In [ ]:
vgg16 = Model(vgg.input,output)

In [ ]:
vgg16.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [ ]:
vgg16.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
vgg16.fit(train,validation_data=test,epochs=2,Batch Size=32)

Epoch 1/2
218/218 [==============================] - 66s 271ms/step - loss: 0.0065 - accuracy: 0.9954 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 2/2
218/218 [==============================] - 55s 253ms/step - loss: 0.0000e+00 - accuracy: 1.0000 - val_loss: 0.0000e+00 - val_accuracy: 1.0000


In [ ]:
vgg16.save('Vgg19.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


# Resnet

In [ ]:
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.layers import Dense,Flatten
from tensorflow.keras.models import Model

In [ ]:
resnet50 = ResNet50(include_top=False,input_shape=(224,224,3))

94765736/94765736 [==============================] - 0s 0us/step


In [ ]:
for layer in resnet50.layers:
  print(layer)

In [ ]:
for layer in resnet50.layers:
  layer.trainable=False

In [ ]:
x = Flatten()(resnet50.output)

In [ ]:
output = Dense(4,activation='softmax')(x)

In [ ]:
resnet50 = Model(resnet50.input,output)

In [ ]:
resnet50.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_2 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 conv1_pad (ZeroPadding2D)   (None, 230, 230, 3)          0         ['input_2[0][0]']             
                                                                                                  
 conv1_conv (Conv2D)         (None, 112, 112, 64)         9472      ['conv1_pad[0][0]']           
                                                                                                  
 conv1_bn (BatchNormalizati  (None, 112, 112, 64)         256       ['conv1_conv[0][0]']          
 on)                                                                                        

In [ ]:
resnet50.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
resnet50.fit(train,validation_data=test,epochs=2)

Epoch 1/2
218/218 [==============================] - 61s 259ms/step - loss: 0.0111 - accuracy: 0.9954 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 2/2
218/218 [==============================] - 53s 245ms/step - loss: 0.0000e+00 - accuracy: 1.0000 - val_loss: 0.0000e+00 - val_accuracy: 1.0000


In [ ]:
resnet50.save('ResNet.h5')

# Inception

In [ ]:
#!pip install tensorflow==2.13

In [ ]:
train = train_datagen.flow_from_directory(trainPath,target_size=(299,299),batch_size=16)
test = test_datagen.flow_from_directory(testPath,target_size=(299,299),batch_size=16)

Found 3476 images belonging to 4 classes.
Found 1307 images belonging to 4 classes.


In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.layers import Dense,Flatten
from tensorflow.keras.models import Model

In [ ]:
InceptionV3 = InceptionV3(include_top=False,input_shape=(299 ,299,3))

87910968/87910968 [==============================] - 0s 0us/step


In [ ]:
for layer in InceptionV3.layers:
  print(layer)

In [ ]:
x = Flatten()(InceptionV3.output)

In [ ]:
output = Dense(4,activation='softmax')(x)

In [ ]:
InceptionV3 = Model(InceptionV3.input,output)

In [ ]:
InceptionV3.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 299, 299, 3)]        0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 149, 149, 32)         864       ['input_3[0][0]']             
                                                                                                  
 batch_normalization (Batch  (None, 149, 149, 32)         96        ['conv2d[0][0]']              
 Normalization)                                                                                   
                                                                                                  
 activation (Activation)     (None, 149, 149, 32)         0         ['batch_normalization[0]

In [ ]:
InceptionV3.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
InceptionV3.fit(train,validation_data=test,epochs=5)

Epoch 1/5
218/218 [==============================] - 146s 474ms/step - loss: 0.0053 - accuracy: 0.9983 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 2/5
218/218 [==============================] - 97s 445ms/step - loss: 0.0000e+00 - accuracy: 1.0000 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 3/5
218/218 [==============================] - 95s 436ms/step - loss: 0.0000e+00 - accuracy: 1.0000 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 4/5
218/218 [==============================] - 95s 433ms/step - loss: 0.0000e+00 - accuracy: 1.0000 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 5/5
218/218 [==============================] - 96s 439ms/step - loss: 0.0000e+00 - accuracy: 1.0000 - val_loss: 0.0000e+00 - val_accuracy: 1.0000


In [ ]:
InceptionV3.save('InceptionV3-covid.h5')

# Xception

In [ ]:
train = train_datagen.flow_from_directory(trainPath,target_size=(299,299),batch_size=16)
test = test_datagen.flow_from_directory(testPath,target_size=(299,299),batch_size=16)

Found 3476 images belonging to 4 classes.
Found 1307 images belonging to 4 classes.


In [ ]:
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.layers import Dense,Flatten
from tensorflow.keras.models import Model

In [ ]:
Xception = Xception(include_top=False,input_shape=(299,299,3))

83683744/83683744 [==============================] - 0s 0us/step


In [ ]:
x = Flatten()(Xception.output)

In [ ]:
output = Dense(4,activation='softmax')(x)

In [ ]:
Xception = Model(Xception.input,output)

In [ ]:
Xception.summary()

Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_4 (InputLayer)        [(None, 299, 299, 3)]        0         []                            
                                                                                                  
 block1_conv1 (Conv2D)       (None, 149, 149, 32)         864       ['input_4[0][0]']             
                                                                                                  
 block1_conv1_bn (BatchNorm  (None, 149, 149, 32)         128       ['block1_conv1[0][0]']        
 alization)                                                                                       
                                                                                                  
 block1_conv1_act (Activati  (None, 149, 149, 32)         0         ['block1_conv1_bn[0][0]'

In [ ]:
Xception.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
Xception.fit(train,validation_data=test,epochs=2)

Epoch 1/2
218/218 [==============================] - 166s 611ms/step - loss: 0.0070 - accuracy: 0.9960 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 2/2
218/218 [==============================] - 119s 546ms/step - loss: 0.0000e+00 - accuracy: 1.0000 - val_loss: 8.4370e-06 - val_accuracy: 1.0000


In [ ]:
Xception.save('Xception-covid.h5')

In [ ]:
InceptionV3.save('InceptionV3-covid.h5')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
model_path = '/content/drive/MyDrive/models/InceptionV3-covid.h5'
InceptionV3.save(model_path)


In [ ]:
from google.colab import files
files.download('InceptionV3-covid.h5')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>